In [ ]:
# Author: M. Riley Owens (GitHub: mrileyowens)


In [ ]:
import sys

import os
import glob

import h5py

import numpy as np

from astropy.io import fits
import astropy.units as u
from astropy.coordinates import SkyCoord

from grizli import utils

import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText

sys.path.append(os.path.abspath('..'))

from mrileyowens.stats import weighted_quantile
from mrileyowens.dja import spectra

In [ ]:
# Set common directories
home = os.getcwd()
data = f'{home}/data'
figs = f'{home}/figs'
results = f'{home}/results'

def select():

    '''
    Plot the SFHs of the 2CSFH BEAGLE models
    '''

    # Get the files containing the EW measurements of the 2CSFH fits
    files = glob.glob(f'{results}/ew/e24_f775w_dropouts_2csfh_no_lya_ews_[!m_uv_e24]*.h5')

    # Get the HDU list of the E24 catalog
    hdul = fits.open(f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits')

    # Make an empty list to contain the indices and coordinates of selected young, weak emission line sources
    idx_e24, coords_e24 = [], []

    # For each EW measurements file
    for i, file in enumerate(files):

        # Open the file
        with h5py.File(file, 'r') as f:

            # For each ID in the file
            for j, id in enumerate(list(f.keys())):

                # Get the probabilities from the posterior
                probs = f[id]['probabilities'][:]

                # Get the emission line EWs measured from the posterior SEDs
                ews_o_iii, ews_h_alpha, ews_h_beta = f[id]['o_iii_ews'][:], f[id]['h_alpha_ews'][:], f[id]['h_beta_ews'][:]

                # Calculate the 84th percentile of the [O III] + H-beta and H-alpha EWs
                p84_h_alpha = weighted_quantile(ews_h_alpha, probs, 0.84)
                p84_o_iii_h_beta = weighted_quantile(ews_o_iii + ews_h_beta, probs, 0.84)

                # If the 84th percentiles of both sets of EWs are sufficiently small, continue
                if p84_h_alpha < 800 and p84_o_iii_h_beta < 800:

                    # Determine the index where the object appears in the E24 catalog
                    idx = np.where(hdul[1].data['ID'] == id)

                    # Get the object's photometry necessary to measure a specific color
                    f150w = (hdul[1].data['NRC_F150W'] * u.nJy).to(u.ABmag)[idx]
                    f200w = (hdul[1].data['NRC_F200W'] * u.nJy).to(u.ABmag)[idx]
                    f277w = (hdul[1].data['NRC_F277W'] * u.nJy).to(u.ABmag)[idx]

                    # Calculate the color, which assesses the photometric slope on either side of the Balmer break at z ~ 6. 
                    # A redder color will correspond to weaker Balmer breaks, and thus young light-weighted ages
                    color = (f200w - f277w) - (f150w - f200w)

                    # If the color is sufficiently red
                    if color < 0.3:

                        # Add the object's coordinates to the list of E24 young, weak emission line sources
                        idx_e24.append(idx[0][0])
                        coords_e24.append([hdul[1].data['RA'][idx][0], hdul[1].data['DEC'][idx][0]])

    # Convert the coordinate list to a NumPy array
    idx_e24, coords_e24 = np.array(idx_e24, dtype=np.int64), np.array(coords_e24, dtype=np.float64)

    # Get the IDs of the young, weak emission line sources in the E24 catalog
    ids = hdul[1].data['ID'][idx_e24]

    # Save the IDs of the young, weak emission line sources in the E24 catalog
    np.savetxt(f'{results}/e24_young_weak_emission_line/ids_e24.txt', ids, fmt='%s')

def f200w():

    '''
    Plot the F200W distribution of the young, weak emission line sources
    '''

    # Get the E24 IDs of the young, weak emission line sources
    ids = np.loadtxt(f'{results}/e24_young_weak_emission_line/ids_e24.txt', dtype=str)

    # Open the HDU list of the E24 catalog
    hdul = fits.open(f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits')

    # Get the indices of the young, weak emission line sources in the E24 catalog
    idx = np.array([np.where(hdul[1].data['ID'] == id)[0][0] for id in ids], dtype=np.int64)

    # Get the F200W photometry of the young, weak emission line sources
    f200w = (hdul[1].data['NRC_F200W'][idx] * u.nJy).to(u.ABmag)

    # Make a new figure to plot the F200W distribution of the young, weak emission line sources
    fig, ax = plt.subplots()

    # Plot the F200W histogram
    ax.hist(f200w.value, bins=20)

    # Label the axes
    ax.set_xlabel('F200W (AB mag.) (E24)')
    ax.set_ylabel('Count')

    # Save the figure
    fig.savefig(f'{figs}/e24_young_weak_emission_line/f200w.png', bbox_inches='tight', dpi=200)

    plt.close('all')

def download_dja_spectra():

    '''
    Download the prism spectra of any coordinate matches to the E24 young, weak emission line sources in the DJA spectroscopic catalog
    '''

    # -------------------------------------------------------------------------------
    # Get the coordinates of the young, weak emission line sources in the E24 catalog
    # -------------------------------------------------------------------------------

    # Get the E24 IDs of the young, weak emission line sources
    ids_e24 = np.loadtxt(f'{results}/e24_young_weak_emission_line/ids_e24.txt', dtype=str)

    # Open the HDU list of the E24 catalog
    hdul_e24 = fits.open(f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits')

    # Get the indices of the young, weak emission line sources in the E24 catalog
    idx_e24 = np.array([np.where(hdul_e24[1].data['ID'] == id)[0][0] for id in ids_e24], dtype=np.int64)

    # Get the coordinates of the sources as a SkyCoord object
    coords_e24 = SkyCoord(ra=hdul_e24[1].data['RA'][idx_e24] * u.deg, dec=hdul_e24[1].data['DEC'][idx_e24] * u.deg)

    # Download the prism spectra of any coordinate matches in the DJA spectroscopic catalog
    spectra(coords_e24, match_strs=['prism'], dir=f'{data}/dja_young_weak_emission_line_spectra')

def plot():

    '''
    Plot the prism spectra of the young, weak emission line sources in the E24 catalog with matches in the DJA spectroscopic catalog
    '''

    # Get the prism spectra
    files = glob.glob(f'{data}/dja_young_weak_line_spectra/*.spec.fits')

    # Open the DJA spectroscopic catalog
    tab = utils.read_catalog(f'{data}/dja_msaexp_emission_lines_v4.4.csv', format='csv')

    # For the FITS file of each prism spectrum
    for i, file in enumerate(files):

        # Get the HDU list
        hdul = fits.open(file)

        # Get the wavelength, flux density, and flux density uncertainty of the spectrum
        w_um = hdul[1].data['wave'] * u.um
        f_ujy = hdul[1].data['flux'] * u.uJy
        f_err_ujy = hdul[1].data['err'] * u.uJy

        # Get the best redshift of the source
        z = tab['z_best'][tab['file'] == os.path.basename(file)].data[0]

        # Make a new figure of the prism spectrum
        fig, ax = plt.subplots()

        # Make a secondary x-axis on the top of the figure to show the rest wavelength
        ax_rest = ax.secondary_xaxis('top', functions=(lambda w: w / (1 + z), lambda w: w * (1 + z)))

        # Plot the spectrum
        ax.plot(w_um, f_ujy, ds='steps-mid', c='black')

        # Plot the uncertainty on the spectrum as a shaded region
        ax.fill_between(w_um, f_ujy - f_err_ujy, f_ujy + f_err_ujy, step='mid', alpha=0.2, color='black')

        # Label the axes
        ax.set_xlabel('Observed wavelength ($\mu$m)')
        ax_rest.set_xlabel('Rest wavelength ($\mu$m)')
        ax.set_ylabel('Flux density (uJy)')

        # Set the y-axis limits
        ax.set_ylim(bottom=0, top=np.min([np.nanmax(f_ujy.value), 20 * np.median(np.abs(f_ujy[~np.isnan(f_ujy)]).value)]))

def sample():

    # Get the prism spectra
    files = glob.glob(f'{data}/dja_young_weak_emission_line_spectra/*.spec.fits')

    mc([fits.open(file)[1].data['wave'] * u.um for file in files], [fits.open(file)[1].data['flux'] * u.uJy for file in files], [fits.open(file)[1].data['err'] * u.uJy for file in files], f'{results}/e24_young_weak_emission_line/mc.h5', N=1000, files=files, overwrite=True)

def balmer():

    files = glob.glob(f'{data}/dja_young_weak_line_spectra/*.spec.fits')

    tab = utils.read_catalog(f'{data}/dja_msaexp_emission_lines_v4.4.csv', format='csv')

    with h5py.File(f'{results}/e24_young_weak_emission_line_bbs.h5', 'w') as f:

        for i, file in enumerate(files):

            hdul = fits.open(file)

            w_um = hdul[1].data['wave'] * u.um
            f_ujy = hdul[1].data['flux'] * u.uJy
            f_err_ujy = hdul[1].data['err'] * u.uJy

            z = tab['z_best'][tab['file'] == os.path.basename(file)].data[0]

            f_mc_ujy = np.random.normal(loc=f_ujy, scale=f_err_ujy, size=(1000, len(f_ujy)))

            bb_ratios = []

            n = 0

            #print(file)

            while n != (len(f_mc_ujy) - 1):

                #mask = f_mc_ujy[n] <= 0.
                
                #f_mc_ujy[n] = np.where(mask, 0, f_mc_ujy[n])

                bb_ratio = np.interp(0.42 * (1 + z) * u.um, w_um, f_mc_ujy[n]) / np.interp(0.35 * (1 + z) * u.um, w_um, f_mc_ujy[n])
                #print(bb_ratio)

                if np.isnan(bb_ratio):

                    #print(n, np.interp(0.42 * (1 + z) * u.um, w_um, f_mc_ujy[n]), np.interp(0.35 * (1 + z) * u.um, w_um, f_mc_ujy[n]))

                    f_mc_ujy[n] = np.random.normal(loc=f_ujy, scale=f_err_ujy, size=len(f_ujy))

                    continue

                bb_ratios.append(bb_ratio)

                n += 1

            print(f'{np.median(bb_ratios):.2f}_-{(np.median(bb_ratios) - np.percentile(bb_ratios, 16)):.2f}^+{(np.percentile(bb_ratios, 84) - np.median(bb_ratios)):.2f}')

def ews():

    lines_dict = {
        'o_iii+h-beta' : [r'[O III] 4959, 5007 $\mathrm{\AA}$', [[4856,5011]], [[4775,4825],[5050,5100]]],
        'h_alpha' : [r'H$\alpha$', [[6558,6568]], [[6478,6528],[6598,6648]]] 
    }

    files = glob.glob(f'{data}/dja_young_weak_line_spectra/*.spec.fits')

    tab = utils.read_catalog(f'{data}/dja_msaexp_emission_lines_v4.4.csv', format='csv')

    with h5py.File(f'{results}/e24_young_weak_emission_line_ews.h5', 'w') as f:

        for i, file in enumerate(files):

            hdul = fits.open(file)

            w_um = hdul[1].data['wave'] * u.um
            f_ujy = hdul[1].data['flux'] * u.uJy
            f_err_ujy = hdul[1].data['err'] * u.uJy

            z = tab['z_best'][tab['file'] == os.path.basename(file)].data[0]

            f_mc_ujy = np.random.normal(loc=f_ujy, scale=f_err_ujy, size=(1000, len(f_ujy)))

            bb_ratios = []

            n = 0

            while n != (len(f_mc_ujy) - 1):

                bb_ratio = np.interp(0.42 * (1 + z) * u.um, w_um, f_mc_ujy[n]) / np.interp(0.35 * (1 + z) * u.um, w_um, f_mc_ujy[n])

                if np.isnan(bb_ratio):

                    f_mc_ujy[n] = np.random.normal(loc=f_ujy, scale=f_err_ujy, size=len(f_ujy))

                    continue

                bb_ratios.append(bb_ratio)

                n += 1

            print(f'{np.median(bb_ratios):.2f}_-{(np.median(bb_ratios) - np.percentile(bb_ratios, 16)):.2f}^+{(np.percentile(bb_ratios, 84) - np.median(bb_ratios)):.2f}')

In [ ]:
select()

In [ ]:
f200w()

In [ ]:
download_dja_spectra()

In [ ]:
plot()

In [ ]:
sample()

In [ ]:
balmer()